# Segmentation Metrics: Confusion Matrix, mIoU and Dice — Solution

**Difficulty**: 🟢 Easy

### Problem Statement

Semantic segmentation models (FCN, DeepLab, SegFormer, Mask2Former, and the segmentation heads of vision foundation models) predict a class for every pixel. Nearly every number reported for them — mIoU on Cityscapes, ADE20K and PASCAL VOC, Dice on medical benchmarks, pixel accuracy — is computed from one $C \times C$ **confusion matrix of pixel counts**. The formulas are easy. What goes wrong are the conventions, which change the headline number without raising any error: which pixels are ignored (the 255 "void" label), what an absent class contributes, and whether you average over images or pool over the dataset.

Build the evaluation toolkit: vectorized, streaming, and exact on those conventions. This problem is about **evaluation**. The differentiable soft-Dice *loss* is in **losses-from-scratch**, and confusion-matrix-based precision/recall for classification is in **classification-metrics**.

**Confusion matrix.** `cm[i, j]` is the number of counted pixels whose ground truth is class $i$ and whose prediction is class $j$. **Rows are ground truth** and columns are predictions, the same as sklearn, torchmetrics and FCN's $n_{ij}$. For class $c$:
$$TP_c = cm_{cc}, \qquad FN_c = \sum_j cm_{cj} - TP_c \ \ (\text{row}), \qquad FP_c = \sum_i cm_{ic} - TP_c \ \ (\text{column}).$$

**Metrics.**
$$\mathrm{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c}, \qquad \mathrm{Dice}_c = \frac{2\,TP_c}{2\,TP_c + FP_c + FN_c} = \frac{2\,\mathrm{IoU}_c}{1 + \mathrm{IoU}_c},$$
$$\text{pixel accuracy} = \frac{\sum_c TP_c}{\sum_{i,j} cm_{ij}}, \qquad \text{mean class accuracy} = \underset{c\,:\,\text{row}_c > 0}{\operatorname{mean}} \ \frac{TP_c}{\sum_j cm_{cj}}, \qquad \mathrm{mIoU} = \underset{c\,:\,TP_c + FP_c + FN_c > 0}{\operatorname{mean}} \ \mathrm{IoU}_c .$$
These are FCN's pixel accuracy, mean accuracy and mean IU (Long et al., 2015, Sec. 5). Dice per class is the F1 score of that class.

**Conventions (graded).**

- **Which pixels count.** A pixel is counted iff its *target* is a class id in $[0, C)$ and is not `ignore_index`. Targets outside $[0, C)$ are void. `ignore_index` only drops pixels by their **target** value. When it is a valid class id, e.g. `ignore_index=0`, that class has no ground-truth pixels left, but a prediction of it on a counted pixel is still an error that lands in its column, so its IoU is 0 if it is ever predicted and NaN otherwise. MMSegmentation does the same; torchmetrics' macro mIoU additionally leaves that class out of the average.
- **Absent classes.** A class with $TP + FP + FN = 0$ (not in the target, never predicted) has **NaN** IoU and Dice, and the means skip it (`torch.nanmean`). A class that is only in the target, or only predicted, scores **0**, not NaN.
- **Dataset-level pooling.** Over a dataset you **sum the confusion matrices of all images and compute the metrics once**. This is how Cityscapes (TP, FP and FN "determined over the whole test set"), PASCAL VOC, ADE20K and MMSegmentation report mIoU. It is **not** the mean of per-image mIoUs. Averaging per image gives every image the same weight whatever its size, and treats a 3-pixel sliver of a class that the model missed in one image (IoU 0) as seriously as a large, well-segmented region in another. Pooling weighs every pixel equally. On the toy dataset below the two differ by 24 points.

---

### Requirements

1. **`confusion_matrix(pred, target, num_classes, ignore_index=255) -> LongTensor`** of shape `(C, C)`, int64.
   - `pred` and `target` are integer label maps of the **same, arbitrary shape** (`(H, W)`, `(B, H, W)`, flattened, ...) and any integer dtype, including the `uint8` you get from PNG masks.
   - Count with **one `torch.bincount`** (or another loop-free counting op). No Python loop over pixels or classes.
   - Drop pixels whose target is outside $[0, C)$ or equals `ignore_index`. `ignore_index=None` means no value is ignored, though the range test still applies.
   - If a **counted** pixel has a prediction outside $[0, C)$, raise `ValueError`. Never look at predictions on dropped pixels.
   - Do not modify the inputs. If nothing is counted, return a zero matrix.
2. **`iou_per_class(cm) -> Tensor`**: `(C,)` float64, NaN for absent classes.
3. **`mean_iou(cm) -> Tensor`**: 0-dim float64, the mean over non-NaN IoUs (NaN if every class is absent).
4. **`dice_per_class(cm) -> Tensor`**: `(C,)` float64, NaN for absent classes.
5. **`pixel_accuracy(cm) -> Tensor`**: 0-dim float64 (NaN for an empty matrix).
6. **`mean_class_accuracy(cm) -> Tensor`**: 0-dim float64, the mean of $TP_c / \text{row}_c$ over the classes that have at least one ground-truth pixel. A class that is predicted but never in the target is skipped.
7. **`SegMetric(num_classes, ignore_index=255)`**, a streaming accumulator:
   - `update(pred, target)` adds that batch's confusion matrix to an **int64** `(C, C)` buffer. Memory stays $O(C^2)$: do not store predictions.
   - `compute()` returns a dict with keys `"confusion_matrix"` (an int64 copy of the summed matrix), `"iou"`, `"mean_iou"`, `"dice"`, `"mean_dice"` (nanmean of Dice), `"pixel_accuracy"` and `"mean_class_accuracy"`, all computed on the **summed** matrix. Calling it does not change the state.
   - `reset()` clears the buffer.

Scalars may be 0-dim tensors or Python floats.

---

### Constraints

- ✅ `torch.bincount`, `reshape`, `diagonal`, `sum`, boolean masks, `torch.nanmean`.
- ❌ No sklearn or torchmetrics (the optional cross-check cell may import sklearn).
- ❌ No Python loop over pixels or classes in `confusion_matrix`. The grader counts torch calls for 64 pixels with 3 classes and for 19,200 pixels with 50 classes, and the two counts must be about the same.
- ❌ No floating-point counts in the accumulator.

---

<details>
  <summary>💡 Hint</summary>

  - Every metric here is a function of the confusion matrix. Which three vectors (one from the diagonal, two from sums) give TP, FP and FN for all classes at once?
  - How can a pair `(target, pred)` with both values in `[0, C)` become a **single** integer in `[0, C*C)`, so that a histogram of those integers *is* the flattened matrix? Which order makes the targets end up on the rows after `reshape(C, C)`?
  - What does `torch.tensor([200], dtype=torch.uint8) * 2` give?
  - What does `torch.tensor(0.0) / 0` give, and what does `torch.nanmean` do with it?
  - Counts add up across batches; metrics do not. What should the accumulator store?

</details>

In [ ]:
import torch

In [ ]:
# A hand-checkable example: C = 3 classes, 255 = void (unlabelled border pixels).
target = torch.tensor([[0, 0, 1, 1],
                       [0, 0, 1, 1],
                       [2, 2, 255, 255],
                       [2, 2, 255, 255]])
pred = torch.tensor([[0, 0, 1, 0],
                     [0, 1, 1, 1],
                     [2, 1, 2, 255],   # a prediction of 255 on a void pixel is never looked at
                     [2, 2, 1, 1]])
# Counted pixels: the 12 with target != 255.
#   target 0 -> predicted [0, 0, 0, 1]   row 0 = [3, 1, 0]
#   target 1 -> predicted [1, 0, 1, 1]   row 1 = [1, 3, 0]
#   target 2 -> predicted [2, 1, 2, 2]   row 2 = [0, 1, 3]
expected_cm = torch.tensor([[3, 1, 0], [1, 3, 0], [0, 1, 3]])


def cm_loop(pred, target, C, ignore_index=255):
    # Brute-force reference: one pixel at a time.
    cm = torch.zeros(C, C, dtype=torch.long)
    for p, t in zip(pred.flatten().tolist(), target.flatten().tolist()):
        if t != ignore_index and 0 <= t < C:
            cm[t, p] += 1
    return cm

In [ ]:
def confusion_matrix(pred, target, num_classes, ignore_index=255):
    """
    Pixel confusion matrix of two integer label maps.

    Args:
        pred: integer tensor of predicted class ids, any shape.
        target: integer tensor of ground-truth class ids, same shape as pred.
        num_classes: C.
        ignore_index: target value to skip (None: skip nothing by value). Target values
            outside [0, C) are void and skipped too.

    Returns:
        (C, C) int64 tensor; cm[i, j] = number of counted pixels with target i and pred j.

    Raises:
        ValueError: if a counted pixel has a prediction outside [0, C).
    """
    C = num_classes
    t = target.reshape(-1).long()          # int64 BEFORE any arithmetic: uint8 would overflow
    p = pred.reshape(-1).long()
    keep = (t >= 0) & (t < C)
    if ignore_index is not None:
        keep &= t != ignore_index          # only the TARGET decides whether a pixel counts
    t, p = t[keep], p[keep]
    if p.numel() and (p.min() < 0 or p.max() >= C):
        raise ValueError(f"predictions must lie in [0, {C}) on counted pixels, "
                         f"got values in [{int(p.min())}, {int(p.max())}]")
    # One integer per (target, pred) pair, one counting pass for the whole matrix.
    return torch.bincount(t * C + p, minlength=C * C).reshape(C, C)

In [ ]:
# Test 1: the confusion matrix
cm = confusion_matrix(pred, target, 3)
assert cm.dtype == torch.long and cm.shape == (3, 3), (cm.dtype, cm.shape)
assert torch.equal(cm, expected_cm), f"expected\n{expected_cm}\ngot\n{cm}"

# rows are ground truth: three class-0 pixels predicted as 1 -> cm[0, 1] == 3
assert confusion_matrix(torch.tensor([1, 1, 1]), torch.tensor([0, 0, 0]), 2).tolist() == [[0, 3], [0, 0]]

# random label maps of several shapes, with void pixels, against the pixel loop
g = torch.Generator().manual_seed(0)
for shape, C in [((50,), 2), ((3, 20, 30), 5), ((2, 2, 7, 9), 19)]:
    t = torch.randint(0, C, shape, generator=g)
    t[torch.rand(shape, generator=g) < 0.1] = 255
    p = torch.randint(0, C, shape, generator=g)
    assert torch.equal(confusion_matrix(p, t, C), cm_loop(p, t, C)), shape
    # uint8 masks (as loaded from PNG): 18 * 19 + 18 does not fit in a byte
    assert torch.equal(confusion_matrix(p.to(torch.uint8), t.to(torch.uint8), C), cm_loop(p, t, C)), "uint8"

# ignore_index variants
assert confusion_matrix(pred, target, 3, ignore_index=0)[0].sum() == 0, "ignore_index=0 drops target-0 pixels"
assert confusion_matrix(pred, target, 3, ignore_index=0)[:, 0].sum() == 1, "...but predictions of 0 still count"
assert confusion_matrix(pred, target, 3, ignore_index=None).sum() == 12, "255 is out of range -> void anyway"

# a bad prediction on a counted pixel is an error
try:
    confusion_matrix(torch.tensor([0, 3]), torch.tensor([0, 1]), 3)
    raise AssertionError("expected ValueError for a prediction of 3 with num_classes=3")
except ValueError:
    pass
print("Test 1 passed: confusion matrix")

In [ ]:
def _tp_fp_fn(cm):
    cm = cm.to(torch.float64)
    tp = cm.diagonal()
    return tp, cm.sum(0) - tp, cm.sum(1) - tp   # columns = predicted, rows = ground truth


def iou_per_class(cm):
    """(C,) float64 IoU = TP / (TP + FP + FN); NaN for a class absent from target and prediction."""
    tp, fp, fn = _tp_fp_fn(cm)
    return tp / (tp + fp + fn)                  # 0 / 0 -> NaN, by design


def mean_iou(cm):
    """0-dim float64: mean IoU over the classes present in target or prediction."""
    return torch.nanmean(iou_per_class(cm))


def dice_per_class(cm):
    """(C,) float64 Dice = 2TP / (2TP + FP + FN) (= F1); NaN for an absent class."""
    tp, fp, fn = _tp_fp_fn(cm)
    return 2 * tp / (2 * tp + fp + fn)


def pixel_accuracy(cm):
    """0-dim float64: correctly labelled counted pixels / counted pixels."""
    cm = cm.to(torch.float64)
    return cm.diagonal().sum() / cm.sum()


def mean_class_accuracy(cm):
    """0-dim float64: mean over classes with ground-truth pixels of TP / (pixels of that class)."""
    cm = cm.to(torch.float64)
    return torch.nanmean(cm.diagonal() / cm.sum(1))

In [ ]:
# Test 2: metrics on the hand example
# TP = [3, 3, 3], rows = [4, 4, 4], cols = [4, 5, 3] -> FN = [1, 1, 1], FP = [1, 2, 0]
iou = iou_per_class(expected_cm)
assert torch.allclose(iou, torch.tensor([3 / 5, 3 / 6, 3 / 4], dtype=torch.float64)), iou
assert abs(float(mean_iou(expected_cm)) - (0.6 + 0.5 + 0.75) / 3) < 1e-12
assert torch.allclose(dice_per_class(expected_cm), torch.tensor([6 / 8, 6 / 9, 6 / 7], dtype=torch.float64))
assert torch.allclose(dice_per_class(expected_cm), 2 * iou / (1 + iou)), "Dice = 2 IoU / (1 + IoU)"
assert abs(float(pixel_accuracy(expected_cm)) - 9 / 12) < 1e-12
assert abs(float(mean_class_accuracy(expected_cm)) - 0.75) < 1e-12

# a 4th class that never occurs and is never predicted: NaN, and the mean ignores it
cm4 = confusion_matrix(pred, target, 4)
assert torch.isnan(iou_per_class(cm4)[3]) and torch.isnan(dice_per_class(cm4)[3])
assert abs(float(mean_iou(cm4)) - float(mean_iou(expected_cm))) < 1e-12, "absent classes must not change mIoU"

# only predicted (class 1) / only in the ground truth (class 0) -> IoU 0, not NaN
only = torch.tensor([[0, 5], [0, 0]])
assert iou_per_class(only).tolist() == [0.0, 0.0]
assert abs(float(mean_class_accuracy(only)) - 0.0) < 1e-12   # class 1 has no GT pixels -> skipped
print("Test 2 passed: IoU, Dice, accuracies")

In [ ]:
class SegMetric:
    """Streaming dataset-level segmentation metrics: keeps only a (C, C) count matrix."""

    def __init__(self, num_classes, ignore_index=255):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.reset()

    def reset(self):
        self.cm = torch.zeros(self.num_classes, self.num_classes, dtype=torch.long)

    def update(self, pred, target):
        self.cm += confusion_matrix(pred, target, self.num_classes, self.ignore_index).to(self.cm.device)

    def compute(self):
        cm = self.cm.clone()
        return {"confusion_matrix": cm,
                "iou": iou_per_class(cm), "mean_iou": mean_iou(cm),
                "dice": dice_per_class(cm), "mean_dice": torch.nanmean(dice_per_class(cm)),
                "pixel_accuracy": pixel_accuracy(cm),
                "mean_class_accuracy": mean_class_accuracy(cm)}

In [ ]:
# Test 3: dataset-level vs per-image mIoU
# Image A: 100 background pixels, except one pixel of class 1 that the model misses.
# Image B: 50 background + 50 class-1 pixels, all correct.
tA = torch.zeros(10, 10, dtype=torch.long); tA[0, 0] = 1
pA = torch.zeros(10, 10, dtype=torch.long)
tB = torch.zeros(10, 10, dtype=torch.long); tB[5:] = 1
pB = tB.clone()

metric = SegMetric(num_classes=2)
for p, t in ((pA, tA), (pB, tB)):
    metric.update(p, t)
res = metric.compute()
per_image = [float(mean_iou(confusion_matrix(p, t, 2))) for p, t in ((pA, tA), (pB, tB))]
print(f"per-image mIoU: {per_image} -> mean {sum(per_image) / 2:.4f}")
print(f"dataset-level mIoU: {float(res['mean_iou']):.4f}")
assert torch.equal(res["confusion_matrix"], torch.tensor([[149, 0], [1, 50]]))
assert abs(float(res["mean_iou"]) - (149 / 150 + 50 / 51) / 2) < 1e-12
assert set(res) >= {"confusion_matrix", "iou", "mean_iou", "dice", "mean_dice",
                    "pixel_accuracy", "mean_class_accuracy"}
assert torch.equal(metric.compute()["confusion_matrix"], res["confusion_matrix"]), "compute() must not change state"
metric.reset()
metric.update(pA, tA)
assert int(metric.compute()["confusion_matrix"].sum()) == 100
print("Test 3 passed: streaming, dataset-level")

In [ ]:
# Optional: cross-check the confusion matrix against sklearn when it is installed.
try:
    from sklearn.metrics import confusion_matrix as sk_confusion_matrix
except ImportError:
    print("sklearn not installed -- skipping the cross-check")
else:
    g = torch.Generator().manual_seed(1)
    t = torch.randint(0, 7, (4, 32, 32), generator=g)
    t[torch.rand(t.shape, generator=g) < 0.1] = 255
    p = torch.randint(0, 7, (4, 32, 32), generator=g)
    keep = t != 255
    ref = sk_confusion_matrix(t[keep].numpy(), p[keep].numpy(), labels=list(range(7)))
    assert (confusion_matrix(p, t, 7).numpy() == ref).all()
    print("matches sklearn.metrics.confusion_matrix (rows = true labels)")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("segmentation-metrics", confusion_matrix=confusion_matrix, iou_per_class=iou_per_class,
      mean_iou=mean_iou, dice_per_class=dice_per_class, pixel_accuracy=pixel_accuracy,
      mean_class_accuracy=mean_class_accuracy, SegMetric=SegMetric)

### Why it works

- **One code per pair.** $(t, p) \mapsto t \cdot C + p$ is a bijection from $[0, C)^2$ to $[0, C^2)$, so a histogram of the codes *is* the confusion matrix, flattened row-major. `minlength=C*C` fixes the length when the last classes never occur, and `reshape(C, C)` puts the target on the rows. That is one $O(N)$ pass and $O(C^2)$ memory, whatever $C$ is.
- **Filter before encoding.** An out-of-range value does not crash. It *aliases*: a prediction of $C$ for target $t$ has code $(t+1)\,C$, which is cell $(t+1, 0)$. That is why targets are masked and the kept predictions are validated before the codes are formed. The cast to int64 comes first for the same reason: in uint8, $18 \cdot 19 + 18 = 360$ wraps to 104.
- **Everything else is the diagonal and two sums.** TP is the diagonal, the row sums are ground-truth areas and the column sums are predicted areas. IoU, Dice and pixel accuracy are unchanged by transposing the matrix (IoU and Dice are symmetric in FP and FN; the trace and the total do not move). Mean class accuracy, per-class precision and recall are not, which is why "rows = ground truth" matters.
- **NaN as "no evidence".** In floating point, $0/0$ is NaN. A class that is neither present nor predicted tells you nothing about the model, so `nanmean` skips it. Scoring it 0 would penalise the model for classes an image does not contain (an ADE20K image typically contains only a small fraction of the 150 classes), and scoring it 1 would reward it.
- **Streaming.** Counts are additive: $cm_{\text{dataset}} = \sum_{\text{batches}} cm_{\text{batch}}$. IoU is a non-linear function of the counts, so the per-batch metrics cannot be averaged. You add the counts and compute once. The same argument makes distributed evaluation exact: all-reduce (sum) the matrices.

### Complexity

- `confusion_matrix`: $O(N)$ time for $N$ pixels, $O(C^2)$ output. The metric functions are $O(C^2)$ (one pass of row and column sums).
- `SegMetric`: $O(C^2)$ memory regardless of dataset size. Storing all predictions and computing at the end gives the same numbers but needs memory linear in the dataset, e.g. ~1 GB of uint8 labels for the 500 Cityscapes validation images at 2048×1024.

### Common bugs

- **Averaging per-image mIoU** instead of pooling the counts. Here this gave 0.75 instead of 0.99.
- **Counting ignored pixels**, e.g. clamping 255 into the last class, or encoding before masking.
- **Absent classes as 0 or 1** instead of NaN plus `nanmean`.
- **A transposed matrix.** It is invisible in IoU and Dice, but wrong for per-class accuracy, precision and recall.
- **Dice computed as IoU**, or as $2 \cdot \mathrm{IoU}$. The correct relation is $2J/(1+J)$.
- **uint8 overflow** in `target * C + pred`, and **float32 accumulators**: float32 counts stop being exact above $2^{24} \approx 16.8$M, and one full-size Cityscapes image has 2.1M pixels.
- **Wrong resolution or interpolation.** Evaluating on downsampled labels, or resizing label maps bilinearly, which invents class ids along boundaries. Upsample the *logits* to the label resolution and resize label maps only with nearest.

### Interview follow-ups

1. **How are mIoU and Dice related?** Per class, $\mathrm{Dice} = 2J/(1+J)$. This is monotone, so per-class rankings agree and $\mathrm{Dice} \ge \mathrm{IoU}$, with equality only at 0 and 1. The *means* are not related by the same formula: $f(J) = 2J/(1+J)$ is concave, so by Jensen's inequality mean Dice $\le 2\,\mathrm{mIoU}/(1+\mathrm{mIoU})$ (it is still $\ge$ mIoU, because Dice $\ge$ IoU for every class). For example, per-class IoUs $\{0, 1\}$ give mIoU 0.5 and mean Dice 0.5, while $f(0.5) = 0.67$. Dice is the per-class F1 of the pixel classifier.
2. **Why do Cityscapes and ADE20K compute mIoU at the dataset level?** Per-image IoU is undefined (0/0) for classes absent from that image, and very noisy for classes that cover a handful of pixels in it. Pooling treats the dataset as one huge image and gives a stable, pixel-weighted estimate. Its weakness is that large objects dominate, which is why Cityscapes also reports **iIoU**: TP and FN pixels are weighted by (average instance size of the class) / (size of their ground-truth instance), so small instances count more. Medical-imaging benchmarks usually do the opposite: Dice is computed per case (one patient volume) and averaged over cases, because every patient counts equally. They then need an explicit rule for a structure that is absent from both the prediction and the ground truth of a case (a common choice is to score it 1).
3. **What is frequency-weighted IoU?** $\mathrm{FWIoU} = \sum_c \frac{t_c}{\sum_k t_k}\,\mathrm{IoU}_c$, with $t_c$ the number of ground-truth pixels of class $c$ (FCN, Long et al. 2015). Frequent classes (road, sky, wall) dominate it, so it behaves more like pixel accuracy and is rarely the headline metric. mIoU exists to give rare classes an equal vote.
4. **What do boundary IoU and the boundary F-score measure that mIoU misses?** For a large object, region IoU is dominated by interior pixels, so sloppy boundaries barely move it. **Boundary IoU** (Cheng et al., CVPR 2021) computes IoU only over the pixels within distance $d$ of each mask's contour, $\frac{|(G_d \cap G) \cap (P_d \cap P)|}{|(G_d \cap G) \cup (P_d \cap P)|}$, with $d$ a fixed fraction of the image diagonal (2% for COCO and ADE20K-like resolutions, 0.5% for the higher-resolution Cityscapes). The **boundary F-score** (Csurka et al., 2013, and DAVIS's contour accuracy $\mathcal{F}$) matches predicted to ground-truth boundary pixels within a distance tolerance and reports the F1 of boundary precision and recall.
5. **What is panoptic quality?** Panoptic segmentation (Kirillov et al., CVPR 2019) gives every pixel a class and, for "things", an instance id. A predicted segment and a ground-truth segment of the same class match if their IoU $> 0.5$, and that matching is unique because segments do not overlap. Then
   $$\mathrm{PQ} = \frac{\sum_{(p,g) \in TP} \mathrm{IoU}(p, g)}{|TP| + \tfrac12|FP| + \tfrac12|FN|} = \underbrace{\frac{\sum_{TP}\mathrm{IoU}}{|TP|}}_{\text{SQ}} \times \underbrace{\frac{|TP|}{|TP| + \tfrac12|FP| + \tfrac12|FN|}}_{\text{RQ}},$$
   computed per class and averaged over classes. SQ is the mean IoU of the matched segments and RQ is a detection F1.
6. **How do you evaluate exactly with distributed data parallel?** Each rank accumulates its own matrix, then you `all_reduce(SUM)` the matrices and compute once. This is exact because counts add, whereas averaging per-rank mIoU is not. Watch `DistributedSampler`: with `drop_last=False` it pads the dataset with repeated samples so every rank gets the same number, which double-counts a few images unless you remove the padding.